# 📓 Cas d'usage IA — [Orientation et tri multimodal des demandeurs d'emploi]

**Auteur** : `Nicolas EVRARD`  
**Promo** : ATOS Atlas IA — Parcours 2 (Pros IT) — Groupe 2 
**Date début** : `10/07/2026`  
**Dernière mise à jour** : `10/07/2026`

---

## 0. Imports & configuration

In [3]:
# Imports standards
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import subprocess
from datetime import datetime
from pathlib import Path

# Reproductibilité
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sys.path.insert(0, "..")  # rend src/ importable depuis notebooks/

# Affichage
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")


# Rechargement automatique des fichiers dans jupyter lors du lancement des fonctions si ces derniers ont été modifiés. 
%load_ext autoreload
%autoreload 2
    
# Scripts outillage
from src.data_loading import charger_donnees #, profiler, taux_de_fraude, fraude_par_type

# Imports ML (à activer au fur et à mesure)
# from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
# from sklearn.preprocessing import StandardScaler, OneHotEncoder
# from sklearn.compose import ColumnTransformer
# from sklearn.pipeline import Pipeline

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 0.5 Traçabilité de la session

In [4]:
# --- Métadonnées du dataset ---
DATASET_NAME = "Sujet Examen CISIA - Promo Upskilling Atlas - mai-oct2026.csv"
DATASET_SOURCE = "data/dataset_trajectoire_emploi_Sujet Examen CISIA - Promo Upskilling Atlas - mai-oct2026 (Session-00279143).csv"        # URL, chemin, ou contact si fourni par le client
DATASET_VERSION = "YYYY-MM-DD"  # date d'extraction OU version explicite
DATASET_SHA1 = "62c207b2eac847bd0fec6ee54a50845f4a324f70"  #


# --- Métadonnées de la session ---
session_date = datetime.now().isoformat(timespec="minutes")
py_version = sys.version.split()[0]

try:
    git_commit = subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL
    ).decode().strip()
except (subprocess.CalledProcessError, FileNotFoundError):
    git_commit = "non disponible (notebook hors repo Git)"

print(f"Date session : {session_date}")
print(f"Python       : {py_version}")
print(f"NumPy        : {np.__version__}")
print(f"Pandas       : {pd.__version__}")
print(f"Git commit   : {git_commit}")
print(f"Dataset      : {DATASET_NAME} (source={DATASET_SOURCE}, version={DATASET_VERSION})")

Date session : 2026-07-11T23:54
Python       : 3.13.13
NumPy        : 2.4.4
Pandas       : 2.3.3
Git commit   : 88e5919
Dataset      : Sujet Examen CISIA - Promo Upskilling Atlas - mai-oct2026.csv (source=data/dataset_trajectoire_emploi_Sujet Examen CISIA - Promo Upskilling Atlas - mai-oct2026 (Session-00279143).csv, version=YYYY-MM-DD)


In [5]:
df = charger_donnees()
display(df.head(5))

print("Cardinalité 'synthese_entretien' :", len(df["synthese_entretien"].unique()))
print("Cardinalité 'classe_retour_emploi' :", len(df["classe_retour_emploi"].unique()))
print("Cardinalité 'est_allocataire' :", len(df["est_allocataire"].unique()))

display (df["synthese_entretien"].unique())

Chargement du CSV : /home/a131270/workspace/briefs/m9/data/dataset_trajectoire_emploi_Sujet Examen CISIA - Promo Upskilling Atlas - mai-oct2026 (Session-00279143).csv


,usager_id,age,niveau_diplome,anciennete_poste_ans,code_rome_vise,code_insee_commune,est_allocataire,nationalite_hors_ue,synthese_entretien,classe_retour_emploi
0,ID_0000,56.0,Bac+2,2.6,D1503,18273,1.0,0,Compétences à réactualiser sur les outils numé...,1
1,ID_0001,46.0,Sans diplôme,10.5,G1302,07240,0.0,0,Cumul de difficultés. Pas de moyen de transpor...,2
2,ID_0002,NaN,Bac,1.4,M1705,01405,NaN,0,Compétences à réactualiser sur les outils numé...,0
3,ID_0003,60.0,Bac,0.2,A1203,63359,1.0,0,Problème ponctuel de mobilité géographique. Zo...,1
4,ID_0004,25.0,Sans diplôme,1.2,N1301,21302,1.0,0,Projet de reconversion à affiner. Besoin d'une...,0


Cardinalité 'synthese_entretien' : 10
Cardinalité 'classe_retour_emploi' : 3
Cardinalité 'est_allocataire' : 3


array(['Compétences à réactualiser sur les outils numériques. Motivation correcte.',
       "Cumul de difficultés. Pas de moyen de transport et garde d'enfants complexe.",
       'Problème ponctuel de mobilité géographique. Zone mal desservie.',
       "Projet de reconversion à affiner. Besoin d'une formation complémentaire.",
       'Excellente présentation, compétences techniques à jour. Reprise rapide visée.',
       'Candidat très dynamique, mobilité nationale validée. Projet clair.',
       "Perte de confiance importante. Risque d'exclusion, barrière de la langue.",
       "Freins périphériques majeurs. Situation d'illettrisme numérique constatée.",
       'Profil autonome, recherche active, aucun frein périphérique identifié.',
       nan], dtype=object)

---
## 1. Cadrage métier & problème

### 1.1 Contexte client

Le client est l'**agence nationale de l’emploi** chargée d’accompagner **les demandeurs d’emploi** et d’adapter les parcours de suivi à leur situation. Son activité repose sur l’analyse de nombreuses informations administratives, socio-professionnelles, géographiques et textuelles recueillies lors des entretiens. Le problème métier consiste à ***identifier rapidement*** les usagers susceptibles de retrouver un emploi à court terme, à moyen terme ou présentant un risque de chômage de longue durée. Une mauvaise orientation peut conduire à un accompagnement inadapté et à une perte de chance pour l’usager. Le recours à l’intelligence artificielle vise donc à fournir **aux conseillers** un outil d’aide à la décision capable d’analyser simultanément des données tabulaires et des synthèses textuelles. L’objectif est d’améliorer la rapidité, la cohérence et la fiabilité de **classification**, tout en maintenant une validation humaine et en respectant les contraintes éthiques et réglementaires.


### 1.2 Énoncé du problème IA

La tâche d’intelligence artificielle consiste à prédire le _délai potentiel de retour à l’emploi d’un demandeur_ à partir de données tabulaires, catégorielles et textuelles. Il s’agit d’un problème de **classification**. L'intelligence artificielle devra classer chaque demandeur selon trois catégories :
- `retour rapide` en moins de 6 mois
- `retour moyen` entre 6 et 12 mois
- `risque de longue durée` au-delà de 12 mois.


### 1.3 Bénéficiaires & impacts

Les utilisateurs directs de la solution sont **les conseillers de l’agence nationale de l’emploi**, qui s’appuient sur la prédiction pour orienter les usagers vers un niveau d’accompagnement adapté. Les responsables d’agence, les équipes métier, les équipes de pilotage, le ministère du travail peuvent également utiliser les résultats agrégés pour suivre les parcours et ajuster les ressources disponibles.

Les personnes indirectement concernées sont principalement **les demandeurs d’emploi**, car la prédiction peut influencer l’intensité de leur accompagnement, l’accès à certaines formations ou la priorité accordée à leur dossier. Une erreur de classification peut donc entraîner un suivi insuffisant, une orientation inadaptée ou une perte de chance. L’administration publique est aussi concernée, puisqu’elle assume la responsabilité de l’usage du système et de ses conséquences.


### 1.4 Critères de succès



| Type | Critère | Cible initiale (client) | Cible révisée après EDA | Justification de la révision |
|---|---|---|---|---|

### 1.5 Risques éthiques & réglementaires anticipés

*TODO*

### 1.6 📝 Synthèse cadrage

*TODO*

---
## 2. Identification & acquisition des données

### 2.1 Sources

| Source | Type | Volumétrie | Format | Accès | Date extraction |
|---|---|---|---|---|---|
| *[ex: SI client]* | *[CSV / SQL / API]* | *[N lignes]* | *[csv, parquet, json]* | *[lecture seule, …]* | `JJ/MM/AAAA` |


In [6]:
# 2.2 Chargement
df = charger_donnees()
display(df.head(5))
df.shape, df.columns.tolist()

Chargement du CSV : /home/a131270/workspace/briefs/m9/data/dataset_trajectoire_emploi_Sujet Examen CISIA - Promo Upskilling Atlas - mai-oct2026 (Session-00279143).csv


,usager_id,age,niveau_diplome,anciennete_poste_ans,code_rome_vise,code_insee_commune,est_allocataire,nationalite_hors_ue,synthese_entretien,classe_retour_emploi
0,ID_0000,56.0,Bac+2,2.6,D1503,18273,1.0,0,Compétences à réactualiser sur les outils numé...,1
1,ID_0001,46.0,Sans diplôme,10.5,G1302,07240,0.0,0,Cumul de difficultés. Pas de moyen de transpor...,2
2,ID_0002,NaN,Bac,1.4,M1705,01405,NaN,0,Compétences à réactualiser sur les outils numé...,0
3,ID_0003,60.0,Bac,0.2,A1203,63359,1.0,0,Problème ponctuel de mobilité géographique. Zo...,1
4,ID_0004,25.0,Sans diplôme,1.2,N1301,21302,1.0,0,Projet de reconversion à affiner. Besoin d'une...,0


((2500, 10),
 ['usager_id',
  'age',
  'niveau_diplome',
  'anciennete_poste_ans',
  'code_rome_vise',
  'code_insee_commune',
  'est_allocataire',
  'nationalite_hors_ue',
  'synthese_entretien',
  'classe_retour_emploi'])

### 2.3 Dictionnaire de variables

| Variable | Description | Type | Plage / valeurs | Cible ? | Sensible ? |
|---|---|---|---|---|---|
| `usager_id` | Identifiant unique du demandeur d'emploi | num/cat | *[min-max ou modalités]* | ❌ | ❌/✅ |
| `age` | Âge de l'usager en années | num/cat | *[min-max ou modalités]* | ❌ | ❌/✅ |
| `niveau_diplome` | Plus haut niveau de diplôme obtenu | num/cat | *[min-max ou modalités]* | ❌ | ❌/✅ |
| `anciennete_poste_ans` | Nombre d'années d'expérience dans le dernier emploi occupé | num/cat | *[min-max ou modalités]* | ❌ | ❌/✅ |
| `code_rome_vise` | Code à 5 caractères du Répertoire Opérationnel des Métiers et des Emplois ciblé par l'usager | num/cat | *[min-max ou modalités]* | ❌ | ❌/✅ |
| `code_insee_commune` | Code officiel géographique INSEE de la commune de résidence | num/cat | *[min-max ou modalités]* | ❌ | ❌/✅ |
| `est_allocataire` | Statut d'indemnisation de l'usager | num/cat | *[min-max ou modalités]* | ❌ | ❌/✅ |
| `nationalite_hors_ue` | *[description]* | num/cat | *[min-max ou modalités]* | ❌ | ❌/✅ |
| `synthese_entretien` | Texte libre contenant les notes textuelles du conseiller lors de l'entretien  | num/cat | *[min-max ou modalités]* | ❌ | ❌/✅ |
| `classe_retour_emploi` | Temps nécessaire au retour à l'emploi | cat | • 0 (Rapide < 6m)<br>• 1 (Moyen 6-12m)<br>• 2 (Risque de longue durée > 12m) | ✅ | ❌/✅ |

### 2.4 📝 Synthèse acquisition

*TODO*